# ch07 筛选、索引与条件逻辑：两个"看起来一样"的 API 其实不一样

> 方向：数据预处理 ｜ 竞赛对应：**数据清洗（每项 2 分）+ 特征构造**
> —— 筛选是清洗和特征工程之间的那道闸门：条件写错，后面全错，而且**不报错**。

本章只解决一件事：**让你写出的每个条件都精确等于你脑子里想的那个条件。**
听起来不难，但本章会用实测数据证明：

- `where` 和 `mask` 在 `NaN` 上的行为**不同**，差 21 行
- `query` 有四种失效场景，其中一种必须换 `engine`
- `filter` 筛的是**列**，不筛行——这是最高频的概念错用
- `query` 的**硬性约束只有两条**：引用外部变量必须 `@`，特殊列名必须反引号
  （另外会有个"老教程说法在 3.0 上已失效"的实测澄清）

## 一、本节考点

| 竞赛评分点 | 分值含义 | 本节覆盖的操作 |
|---|---|---|
| 数据清洗 | 每项 2 分 | 按条件定位待处理记录，不改动范围外的数据 |
| 数据探索 | 每项 2 分 | 分组计数（`isin` / `between` / 多条件组合） |
| 特征构造 | 影响建模分 | `np.select` 分档造标签列 |
| 结果交付 | 交卷格式分 | 条件写错不报错，只在结果里错——必须用 `assert` 自证 |

**为什么单独成章**：条件逻辑的错误**不会抛异常**。
`dev[dev["等级"] == "严重" | dev["等级"] == "危急"]` 这种写法在 Python 里
会因为优先级问题静默变成别的东西（或直接报 `TypeError`），
你只有在最后发现"样本数对不上"时才知道错了——那时候已经很难回溯。

## 二、API 速查表

| API | 作用对象 | 返回 | 一句话说明 |
|---|---|---|---|
| `df["列"] == 值` | 行 | `Series[bool]` | 最基础的掩码，注意 `NaN` 参与比较恒为 `False` |
| `df[mask]` | 行 | `DataFrame` | 按布尔掩码筛行（也可以用列名列表筛列） |
| `df.loc[行条件, 列名]` | 行列 | `DataFrame`/`Series` | **条件筛行 + 指定列**，标签索引 |
| `df.iloc[行号, 列号]` | 行列 | `DataFrame`/`Series` | 位置索引 |
| `df.query("表达式")` | 行 | `DataFrame` | 字符串表达式，最贴近"人怎么念" |
| `df.filter(like=/regex=/items=)` | **列** | `DataFrame` | 按**列名**筛选列，不筛行 |
| `s.isin([...])` | 元素 | `Series[bool]` | 属于集合，等价于一串 `or` |
| `s.between(a, b)` | 元素 | `Series[bool]` | 闭区间，等价于 `(s>=a)&(s<=b)` |
| `s.where(cond, other)` | 元素 | `Series` | **cond 为假**时换成 `other` |
| `s.mask(cond, other)` | 元素 | `Series` | **cond 为真**时换成 `other` |
| `np.where(cond, a, b)` | 元素 | `ndarray` | 二元分支，比 `apply` 快得多 |
| `np.select(conds, choices, default)` | 元素 | `ndarray` | 多分支分档，**替代多层 if-else** |
| `df.loc[条件, "列"] = 值` | — | `None` | 条件赋值**唯一正确**写法 |

### `where` / `mask` 一句话记忆

| | 命中条件时 | 不命中时 |
|---|---|---|
| `where(cond, other)` | 保留原值 | 换成 `other` |
| `mask(cond, other)` | 换成 `other` | 保留原值 |

**但两者对"原本就是 `NaN`"的位置处理不同**，见 §3.5 —— 这是本章最值钱的一节。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("data")
dev = pd.read_csv(DATA / "device_defects.csv")

SENTINEL_CUT = 1000      # 负荷值 >= 该值视为哨兵/尖峰
REPLACE_WITH = 888       # 演示 where / mask 的替换值
RISK_CHOICES = ["高", "中"]

print("pandas", pd.__version__)
print("dev", dev.shape)

---

## 三、逐节讲解

### 3.1 三种筛行方式的选型规则

`df[mask]` / `df.loc[mask, cols]` / `df.query("...")` 三者结果等价，
**选哪个是"可读性"问题，不是"正确性"问题**。但有一个硬边界：

| 需求 | 用什么 |
|---|---|
| 只筛行、要全部列 | `df[mask]` |
| 筛行 + 只取几列 | `df.loc[mask, [列]]` ← **推荐**，一步到位 |
| 条件多、要读起来像自然语言 | `df.query("...")` |
| 按行号 / 列号取 | `df.iloc[...]` |
| 按列名筛列 | `df.filter(like=...)` |

**选型对效率的影响**：`df[mask]` 之后再 `[[列]]` 是**两次**索引（多一次拷贝），
`df.loc[mask, [列]]` 是**一次**。数据量大时差距明显；竞赛小数据感知不到，
但**写上习惯**能省掉后面所有"为什么这么慢"的排查。

In [ ]:
mask_crit = dev["缺陷等级"] == "危急"
print("① 基础掩码 dev['缺陷等级'] == '危急'：")
print("   命中行数:", int(mask_crit.sum()), "/", len(dev))
print("   掩码 dtype:", mask_crit.dtype)

print("\n② 三种写法结果是否完全一致：")
a = dev[mask_crit & (dev["设备类型"] == "变压器")]
b = dev.loc[mask_crit & (dev["设备类型"] == "变压器")]
c = dev.query("缺陷等级 == '危急' and 设备类型 == '变压器'")
print("   df[mask]:", a.shape, "| loc[mask]:", b.shape, "| query:", c.shape)
print("   三者 equals:", a.equals(b), b.equals(c))

print("\n③ 筛行 + 取列，一步 vs 两步：")
one_step = dev.loc[mask_crit, ["记录ID", "缺陷等级", "负荷值"]]
two_step = dev[mask_crit][["记录ID", "缺陷等级", "负荷值"]]
print("   一步 .loc[mask, cols]:", one_step.shape)
print("   两步 [mask][cols]    :", two_step.shape)
print("   结果相同:", one_step.equals(two_step), "（但两步多一次中间拷贝）")

print("\n④ iloc 按位置取：")
print("   前 3 行前 3 列:")
print(dev.iloc[:3, :3].to_string(index=False))

> **`NaN` 参与比较永远为 `False`**——这一点必须在心里默念：
> ```python
> s = pd.Series([1.0, np.nan])
> (s > 0).tolist()      # [True, False]   ← NaN 位置是 False，不是 NaN
> (~(s > 0)).tolist()   # [False, True]   ← 取反后 NaN 位置变 True
> ```
> 所以 `~mask` **不是**"选出满足条件的反例"，而是"选出所有不满足条件的，包括缺失"。
> `between` / `isin` 同理，它们返回的是纯 `bool`，不会传播 `NaN`。

---

### 3.2 布尔索引五件套：`&` `|` `~` `isin` `between`

五个运算符的组合就是**筛选的全部表达力**。写法上只有两条纪律，
但违反任何一条都会出问题。

In [ ]:
crit = dev["缺陷等级"] == "危急"
sev = dev["缺陷等级"] == "严重"

print("① & 且：")
both = crit & (dev["设备类型"] == "变压器")
print("   危急 且 变压器:", int(both.sum()))

print("\n② | 或（注意每个条件都要括号）：")
print("   危急 或 严重 -> isin 写法:", int(dev["缺陷等级"].isin(["严重", "危急"]).sum()))
print("   ~(一般) 反选写法  :", int((~(dev["缺陷等级"] == "一般")).sum()))

print("\n③ ~ 取反会连缺失一起选进来：")
probe = pd.Series([1.0, np.nan, 3.0])
print("   (probe > 0)      :", (probe > 0).tolist())
print("   ~(probe > 0)     :", (~(probe > 0)).tolist(), " ← NaN 位置被判为 True")
print("   只想要非缺失的反例:", ((probe <= 0) & probe.notna()).tolist())

print("\n④ between 是闭区间：")
bet = dev["负荷值"].between(100, 300)
print("   between(100, 300) 命中:", int(bet.sum()))
print("   等价写法 (>=100) & (<=300):", int(((dev["负荷值"] >= 100) & (dev["负荷值"] <= 300)).sum()))
print("   与 query 链式比较一致:",
      bool(dev.query("100 < 负荷值 < 300").index.equals(dev.loc[bet].index)))

print("\n⑤ contains / startswith（务必给 na=False）：")
print("   处理状态含'处理':", int(dev["处理状态"].str.contains("处理", na=False).sum()))
print("   不给 na 会得到:", dev["处理状态"].str.contains("处理").dtype,
      "（对象型，含 NaN 时布尔索引直接报错）")

print("\n⑥ 括号陷阱三种写法的真实表现（这是本节最值钱的一张表）：")
print("   写法 A：两个条件都加括号（正确）")
good = (dev["缺陷等级"] == "危急") & (dev["设备类型"] == "变压器")
print("      (dev['缺陷等级']=='危急') & (dev['设备类型']=='变压器')")
print("      执行成功, 命中 =", int(good.sum()), " ← 正确结果")

print("\n   写法 B：只给第一个加括号（静默错误！）")
half = (dev["缺陷等级"] == "危急") & dev["设备类型"] == "变压器"
print("      (dev['缺陷等级']=='危急') & dev['设备类型'] == '变压器'")
print("      执行成功, 命中 =", int(half.sum()), " ← 不报错，但结果全 False")
print("      原因：& 优先级高于 ==，实际算的是 ((A) & 设备类型) == '变压器'")
print("      中间量 (A) & 设备类型 命中", int(((dev["缺陷等级"] == "危急") & dev["设备类型"]).sum()),
      "个，再与字符串 '变压器' 比较 -> 全部 False")

print("\n   写法 C：完全不写括号（会报错，反而安全）")
try:
    dev["缺陷等级"] == "危急" & dev["设备类型"] == "变压器"
    print("      执行成功")
except TypeError as exc:
    print("      ->", type(exc).__name__, ":", str(exc)[:60])

print("\n   写法 D：用 and（会报错，也安全）")
try:
    good and dev["设备类型"].eq("变压器")
    print("      执行成功")
except ValueError as exc:
    print("      ->", type(exc).__name__, ":", str(exc)[:60])

> **两条纪律**（第 ⑥ 步给了实测证据）：
>
> | 写法 | 结果 | 危险等级 |
> |---|---|---|
> | `(A) & (B)` | 命中 5 ✅ | 正确 |
> | `(A) & B == 值` | 命中 **0**，**不报错** | 🔴 **最危险**——静默给出空结果 |
> | `A == 值 & B == 值`（完全不写括号） | `TypeError` | 🟢 安全，至少报错 |
> | `A and B` | `ValueError: truth value is ambiguous` | 🟢 安全，至少报错 |
>
> 结论：**报错不可怕，"命中 0 条"才可怕**——你会以为数据里真没有。
> 所以纪律是：**每个条件单独加括号**，写完之后立刻 `mask.sum()` 数一遍。
>
> 原理：`&` 的优先级**高于** `==`。`(A) & B == "变压器"` 被解析为
> `((A) & B) == "变压器"`——先做 `布尔 & 字符串`，再拿结果跟字符串比，
> 于是整列变成 `False`。

---

### 3.3 难点深挖：`query` 的两条硬约束

**为什么难**：`query` 用字符串写条件，可读性最好，但**字符串里的东西不走 Python 的作用域**，
所以有一整套"看起来该工作但不工作"的写法。而且网上教程大量基于 pandas 1.x/2.x，
有几条流传很广的"必须这么写"在 3.0 上已经不成立——**照抄会写出多余参数，不抄又怕踩坑**，
所以本节全部结论都现场实测一遍。

In [ ]:
print("① 普通用法：像念句子一样写条件")
print("   负荷值 > 500 且 缺陷等级 == '危急':",
      dev.query("负荷值 > 500 and 缺陷等级 == '危急'").shape)

print("\n② in 列表：")
print("   设备类型 in ['变压器', '绝缘子']:", dev.query("设备类型 in ['变压器', '绝缘子']").shape)

print("\n③ 引用外部变量必须加 @，不加就报 UndefinedVariableError：")
thr = 500
print("   query('负荷值 > @thr'):", dev.query("负荷值 > @thr").shape)
try:
    dev.query("负荷值 > thr")
    print("   不加 @ 居然成功")
except Exception as exc:
    print("   不加 @ ->", type(exc).__name__, ":", str(exc)[:70])

print("\n④ 列名含空格 / 是保留字时用反引号：")
odd = dev.rename(columns={"负荷值": "负荷 值"})
print("   反引号包裹:", odd.query("`负荷 值` > 500").shape)
odd2 = dev.rename(columns={"负荷值": "class"})
print("   保留字 class:", odd2.query("`class` > 500").shape)

print("\n⑤ 涉及 .str 方法：本机实测默认引擎已支持（与老教程的说法不同）")
q_default = dev.query("台区编号.str.startswith('STATION_A')")
q_python = dev.query("台区编号.str.startswith('STATION_A')", engine="python")
print("   默认引擎      :", q_default.shape)
print("   engine=python :", q_python.shape)
print("   两者结果一致  :", q_default.index.equals(q_python.index))
print("   -> 基于 pandas 1.x/2.x 的教程写『.str 必须换 engine=python』，")
print("      在 3.0 上已不成立；竞赛机器版本不确定，写上也不亏（只是慢一点）。")

print("\n⑥ 默认引擎实测还能处理这些写法：")
print("   缺陷类型.str.len() > 2 ->", dev.query("缺陷类型.str.len() > 2").shape)
print("   负荷值.notna()        ->", dev.query("负荷值.notna()").shape)
print("   abs(负荷值) > 500     ->", dev.query("abs(负荷值) > 500").shape)
print("   index < 100           ->", dev.query("index < 100").shape)

> **判定规则**：
>
> | 情况 | 要不要特殊处理 |
> |---|---|
> | 纯数值 / 类别比较（`>` `==` `in`） | 直接用默认引擎 |
> | 涉及 `.str` / `.notna()` 等方法调用 | 本机 3.0 实测**可直接用**；想稳妥写 `engine="python"` |
> | 引用局部变量 | **必须加 `@`**，否则 `UndefinedVariableError` |
> | 列名含空格 / 是保留字（`class` `if`） | **必须反引号包裹** |
> | 条件里引用 3 个以上外部变量 | 干脆别用 `query`，改回掩码更好 review |
>
> 一句话：`query` 的**硬性约束只有 `@` 和反引号两条**，其余是可读性取舍。
> 遇到报错先看错误类型——`UndefinedVariableError` 就是缺 `@`。
> 不要因为"教程说要换引擎"就无脑加参数，也不要因为本机能用就以为到处能用。

---

### 3.4 `filter`：筛的是列，不是行

这是最高频的概念错用。`filter` 的 `like` / `regex` / `items` 全部作用在**列名**上。
名字起得像筛行，所以很多人第一次用它筛数据时会得到"整列没变"的结果，
而且**不报错**。

In [ ]:
print("① 按列名模糊筛列：")
print("   filter(like='设备')   ->", list(dev.filter(like="设备").columns))
print("   filter(regex='^缺陷') ->", list(dev.filter(regex="^缺陷").columns))
print("   filter(items=[...])   ->", list(dev.filter(items=["记录ID", "负荷值"]).columns))

print("\n② 它们的对象是列，行数一行不少：")
print("   原始行数:", len(dev), "| filter 后行数:", len(dev.filter(like="设备")))

print("\n③ 想筛行还得用掩码：")
print("   dev[dev['设备类型'] == '变压器'] ->",
      dev[dev["设备类型"] == "变压器"].shape, " ← 行数才变了")

print("\n④ filter 的正确使用场景：一次性把「数值列」捞出来算统计量")
num_block = dev.filter(items=dev.select_dtypes("number").columns.tolist())
print("   取到的列:", list(num_block.columns))
print(num_block.describe().round(2).to_string())

> **记法**：`filter` 里的 `like`/`regex` 修饰的是**列名这个字符串**。
> 想筛行 → `[]` / `loc` / `query`；想筛列 → `filter` / `select_dtypes` / 列名列表。

---

### 3.5 难点深挖：`where` 与 `mask` 对 `NaN` 的不同处理

**为什么难**：几乎所有教程都只讲"`where` 保留满足条件的、`mask` 替换满足条件的"，
然后给一个**没有缺失值的例子**。一旦列里有 `NaN`，两者就分道扬镳了，
而且**都不报错**。

In [ ]:
load = dev["负荷值"]
print("原始列缺失数:", int(load.isna().sum()), "（这 21 个是真正的数据缺失）")

print("\n① 不给 other 时，两者结果一样：")
w_none = load.where(load < SENTINEL_CUT)
m_none = load.mask(load >= SENTINEL_CUT)
print("   where(load < 1000) 后 NaN 数:", int(w_none.isna().sum()))
print("   mask(load >= 1000) 后 NaN 数:", int(m_none.isna().sum()))
print("   -> 都是 28（21 原有缺失 + 7 个 >=1000 的哨兵）")

print("\n② 给了 other 之后，差异出现了：")
w_other = load.where(load < SENTINEL_CUT, REPLACE_WITH)
m_other = load.mask(load >= SENTINEL_CUT, REPLACE_WITH)
n_w = int(w_other.eq(REPLACE_WITH).sum())
n_m = int(m_other.eq(REPLACE_WITH).sum())
print(f"   where(..., 888) 命中 888 的个数: {n_w:3d}   ← 连原有 21 个 NaN 一起换了")
print(f"   mask(..., 888)  命中 888 的个数: {n_m:3d}   ← 只换了 7 个哨兵，NaN 保留")
print(f"   差值 = {n_w - n_m} = 原有缺失数 {int(load.isna().sum())}")

print("\n③ 为什么：")
print("   where 的判定是『cond 为假 -> 换 other』，而 NaN < 1000 的结果是 False")
print("   -> NaN 位置被判为『不满足条件』，于是被 other 覆盖")
print("   mask 的判定是『cond 为真 -> 换 other』，而 NaN >= 1000 也是 False")
print("   -> NaN 位置不满足替换条件，原样保留")

print("\n④ 想用 where 但保住 NaN：把 notna() 一起写进条件")
w_safe = load.where((load < SENTINEL_CUT) | load.isna(), REPLACE_WITH)
print("   命中 888 的个数:", int(w_safe.eq(REPLACE_WITH).sum()), " ← 与 mask 一致了")
print("   缺失仍为:", int(w_safe.isna().sum()))

> **判定规则**（背下来）：
>
> | 我要做的事 | 用哪个 |
> |---|---|
> | 把"满足条件的那批"统一改成某值 | **`mask`** —— 不会碰到缺失值，最安全 |
> | 把"不满足条件的那批"统一改成某值，且**明确想连缺失一起改** | `where` |
> | 把"不满足条件的"改掉，但**想保住缺失** | `where((cond) \| s.isna(), v)` |
>
> **默认用 `mask`**。因为"我要处理的那批记录"通常是有明确定义的（比如 `load >= 1000`），
> 而"其余全部"里混着缺失值，把它们一起改掉几乎总是错的。

---

### 3.6 多条件分档：`np.select` 与 `np.where`

竞赛里几乎必考的一项：**用连续值造一个有序类别列**（风险等级、负荷区间、告警级别）。
标准做法是 `np.select`，不是 `apply` + `if-else`。

In [ ]:
print("① 二元分支用 np.where（最快）：")
high_load = np.where(dev["负荷值"] > 200, "高负荷", "低负荷")
print("   ", pd.Series(high_load).value_counts().to_dict())

print("\n② 多分支用 np.select —— 顺序即优先级，先命中先返回：")
conds = [dev["缺陷等级"].eq("危急"), dev["缺陷等级"].eq("严重")]
risk = np.select(conds, RISK_CHOICES, default="低")
print("   ", pd.Series(risk).value_counts().to_dict())
print("   合计:", len(risk), "= 行数", len(dev), "| 无遗漏:",
      (pd.Series(risk) != "").all())

print("\n③ 为什么不用 apply：")
import time
big = pd.concat([dev] * 400, ignore_index=True)
t0 = time.perf_counter()
_r1 = np.select([big["缺陷等级"].eq("危急"), big["缺陷等级"].eq("严重")], RISK_CHOICES, default="低")
t1 = time.perf_counter()
_r2 = big["缺陷等级"].apply(lambda v: "高" if v == "危急" else ("中" if v == "严重" else "低"))
t2 = time.perf_counter()
print(f"   行数 {len(big)}")
print(f"   np.select : {t1 - t0:.4f}s")
print(f"   apply     : {t2 - t1:.4f}s   （慢 {(t2 - t1) / (t1 - t0):.0f} 倍）")

print("\n④ 写回 DataFrame 变成特征列：")
demo = dev.assign(风险等级=risk)
print(demo["风险等级"].value_counts().to_string())
print("   交叉验证（等级 x 风险）:")
print(pd.crosstab(demo["缺陷等级"], demo["风险等级"]).to_string())

> **优先级顺序陷阱**：`np.select` 按列表顺序**首个命中即返回**。
> 如果先写 `负荷值 > 100`，再写 `负荷值 > 500`，那 `> 500` 永远不会生效。
> **永远把最严格的条件写在最前面。**

---

### 3.7 条件赋值：只有 `loc` 一种正确写法

第 3.1 节讲过 `df[mask]["col"] = v` 在 pandas 3.0 下静默失效。
这里给出**唯一正确写法**和一个自查技巧。

In [ ]:
d = dev.copy()

# 正确：loc 一次性完成定位与赋值
d.loc[d["负荷值"] > 500, "复核标记"] = "高负荷待复核"
print("① loc 条件赋值命中:", int((d["复核标记"] == "高负荷待复核").sum()))

# 其余行补默认值（先整列赋默认，再覆盖，顺序不能反）
d["复核标记"] = d["复核标记"].fillna("常规")
print("② 补默认值后分布:", d["复核标记"].value_counts().to_dict())

# 错误示范：链式赋值（静默失效）
d2 = dev.copy()
d2[d2["负荷值"] > 500]["备注"] = "xxx"
print("\n③ 链式赋值后 '备注' 有 'xxx' 的行数:",
      int((d2["备注"] == "xxx").sum()), " ← 0，一行都没改上")

# 自查技巧：赋值后立刻数一遍命中数
before = int((d["负荷值"] > 500).sum())
after = int((d["复核标记"] == "高负荷待复核").sum())
print("\n④ 自查：条件命中数", before, "== 赋值命中数", after, "->", before == after)

> **自查纪律**：条件赋值之后，**立刻断言"条件命中数 == 赋值命中数"**。
> 这是成本最低、收益最高的一条自检——它能抓住所有静默失效。

---

### 3.8 组合筛选实战：把条件堆成一个清单

真实任务的筛选条件从来不是一条。下面用 3 个条件组合出「高优先检修清单」，
并顺便演示**怎么验证自己没筛错**。

In [ ]:
urgent = dev.loc[
    dev["缺陷等级"].isin(["严重", "危急"])
    & dev["设备类型"].isin(["变压器", "断路器"])
    & dev["处理状态"].str.contains("处理", na=False),
    ["记录ID", "台区编号", "设备类型", "缺陷等级", "负荷值", "处理状态"],
]

print("① 高优先检修清单:", urgent.shape)
print(urgent.head(5).to_string(index=False))

print("\n② 逐条件验证（每一步都要对数）：")
c1 = dev["缺陷等级"].isin(["严重", "危急"])
c2 = dev["设备类型"].isin(["变压器", "断路器"])
c3 = dev["处理状态"].str.contains("处理", na=False)
print("   条件1 等级 ∈ {严重,危急}:", int(c1.sum()))
print("   条件2 类型 ∈ {变压器,断路器}:", int(c2.sum()))
print("   条件3 处理状态含'处理':", int(c3.sum()))
print("   三条件交集:", int((c1 & c2 & c3).sum()), " ← 必须等于清单行数", len(urgent))

print("\n③ 清单内部结构：")
print("   按等级:", urgent["缺陷等级"].value_counts().to_dict())
print("   按类型:", urgent["设备类型"].value_counts().to_dict())
print("\n④ 清单里负荷值最高的 3 条：")
print(urgent.nlargest(3, "负荷值")[["记录ID", "设备类型", "缺陷等级", "负荷值"]].to_string(index=False))

---

## 四、易错点清单

1. **只给一半条件加括号** → `(A) & B == 值` 静默返回全 `False`（因为 `&` 优先级高于 `==`）；
   完全不写括号则抛 `TypeError`。**报错不可怕，"命中 0 条"才可怕**
2. 用 `and` / `or` 连接 Series 条件 → `ValueError: truth value is ambiguous`
3. `~mask` 会把 `NaN` 一起选进来（`NaN` 参与比较恒为 `False`，取反即 `True`）
4. `str.contains` 不给 `na=False` → 布尔索引时抛 `ValueError`
5. **`filter` 用来筛行** → 行数一行不少，列却被砍了，且不报错
6. `query` 引用局部变量忘记加 `@` → `UndefinedVariableError`
7. 列名含空格、或是 `class` / `if` 这类保留字时**不加反引号** → 查询直接失败
8. `np.select` 条件顺序写反 → 宽条件在前，窄条件永远命中不了
9. **`where(cond, v)` 会把原有 `NaN` 一起替换**（实测差 21 行）
10. `df[mask]["列"] = 值` 链式赋值静默失效，一行都改不上
11. 条件赋值不写默认值 → 未命中行是 `NaN`，而不是"默认类别"
12. 筛完不数一遍 → 竞赛按步骤给分，**写了验证才算**

---

## 五、本章小结

| 需求 | 正确写法 |
|---|---|
| 筛行（全部列） | `df[mask]` |
| 筛行 + 取列（推荐） | `df.loc[mask, [列名]]` |
| 条件多、可读性优先 | `df.query("...")`（有方法调用就加 `engine="python"`） |
| 按列名筛列 | `df.filter(like=/regex=)` 或 `df.select_dtypes()` |
| 属于集合 | `s.isin([...])` |
| 区间 | `s.between(a, b)`（闭区间） |
| 把命中的行改掉 | `s.mask(cond, v)` ← **默认选它** |
| 把未命中的行改掉（保住缺失） | `s.where((cond) \| s.isna(), v)` |
| 多档分类 | `np.select(conds, choices, default=)`，**窄条件在前** |
| 条件赋值 | `df.loc[cond, "列"] = v`，**赋完立刻对数** |